## GUC Tariff Design - Exploring the Code ##

In [ ]:
# Package Imports
import os
import sys
import time
import json
import logging
import warnings
import pandas as pd
import numpy as np
import geopandas as gpd
import polars as pl
import matplotlib.pyplot as plt
from datetime import datetime

# Data Setup and Location Information
# US Shapefile data is intentionally untracked in this code due to size, replace the pathway below with the location of your shapefile data
CA_gdf = gpd.read_file("/Users/neil.stein/Documents/General Data Storage/cb_2024_us_all_500k/cb_2024_us_county_500k.zip")
CA_gdf = CA_gdf[CA_gdf["STATEFP"] == "06"]

# Confirming the CRS and storing for future use to set other data sources to match
CA_crs = CA_gdf.crs
print(CA_crs)

EPSG:4269


In [ ]:
# Segment Script (Copy from Al)
# Inverting the code to have the configur-able segments at the top
state = "CA"
utility = "Pacific Gas and Electric Company"
gas_utility = ""

# Defining target data columns
NON_METADATA_COLS = [
    "elec_utility",
    "elec_weight",
    "gas_utility",
    "gas_weight",
    "elec_county_coverage",
    "gas_county_coverage",
    "SP_elec",
    "SP_gas",
    "electricity.total",
    "natural_gas.total",
    "heating_pct",
    "cooling_pct",
    "hot_water_pct",
    "cooking_pct",
    "others_pct",
    "total_kwh"
]


# Function building for data organization
def get_segment(state, utility="", gas_utility="", upgrade = 0, sample=None, **kwargs):
    """
    State is required, utility and gas_utility are recommended. The rest of the arguments are optional.

    Provide each segment with a *list* of options applicable to that segment.
    
    It works by finding the all entries that contain any string provided as input for a given segment. For
    example, providing "Electric" for heating_type will include all electric resistance heating systems
    (baseboards, boilers, etc). Below is a list of all available options for each segment, but there are important
    options that can be matched that don't appear in this list, such as the county or city, as well as a few others.

    "heating_type":         "Electric | Natural Gas | Propane | Oil | Shared | None", (k)\n
    "building_type":        "SF | Small MF | Large MF | Mobile", (e) \n
    "wh_type":              "Heat Pump | Electric | Natural Gas | Propane | Oil", (k)\n
    "area":                 "0-1499 | 1500-2499 | 2500-3999 | 4000+", (e)\n
    "income":               "Not Available | Low Income (<40,000) | Moderate Income (40,000-99,999) | High Income (>100,000)", (e)\n
    "climate_zone":         "Cold | Hot-Dry | Hot-Humid | Marine | Mixed-Dry | Mixed-Humid | Very Cold", (e)\n
    "vintage":              "<1980 | 1980-2000 | 2000-2010 | >2010", (e)\n
    "insulation_level":     "Good Insulation | Average Insulation | Poor Insulation", (e)\n
    "has_solar":            "Yes | No" (e)

    (e) = exact match only, (k) = keyword match

    """
    df = pl.read_parquet("outputs/RMI_2024_7.5_blk_grp_pop_cust_cnt.parquet").filter((pl.col("state").str.contains(state, literal=True)))

    # Handle utility as string or list
    if isinstance(utility, str):
        df_util = df.filter(pl.col("elec_utility").str.contains(utility, literal=True))
    else:
        df_util = df.filter(pl.col("elec_utility").str.contains_any(utility, literal=True))

    df_util = df_util.with_columns(
        bldg_id = pl.col("bldg_id").cast(str),
        zip_code = pl.col("zip_code").cast(str).str.zfill(5)
    )

    # Electric Utility name invalid for state
    if df_util.is_empty():
        raise Exception(f"Utility name invalid. Has to be one of the following: {', '.join(df['elec_utility'].drop_nulls().unique().sort())}")
    
    # Handle gas utility as string or list
    if isinstance(gas_utility, str):
        df_util = df_util.filter(pl.col("gas_utility").str.contains(gas_utility, literal=True))
    else:
        df_util = df_util.filter(pl.col("gas_utility").str.contains_any(gas_utility, literal=True))
    
    # Gas Utility name invalid for state or no buildings with the selected electric utility are served gas by the selected gas utility
    if df_util.is_empty():
        raise Exception(f"Gas utility name invalid or no buildings with the selected electric utility are served gas by the selected gas utility. Choose one of the following: {', '.join(df['gas_utility'].unique().sort().to_list())}?")

    # Filtering
    for col_name, values in kwargs.items():
        if not values:
            continue  # Skip empty values
        if col_name not in df.columns:
            raise Exception(f"{col_name} is not a valid name. Allowed names are {', '.join(df.columns)}")

        if all(~df[col_name].str.contains_any(values) if type(values[0]) is str else ~df[col_name].is_in(values)):
            raise Exception(f"One of {values} is not valid for {col_name}. Allowed options are {df[col_name].unique().sort().to_list()}")

        df_util = df_util.filter(pl.col(col_name).str.contains_any(values)) if type(values[0]) is str else df_util.filter(pl.col(col_name).is_in(values))
    
    if df_util.is_empty():
        raise Exception("Segment is too narrow for this utility and has no samples in ResStock.\nCheck sgements_by_utility folder for avilable segments for this utility.")

    if sample is not None:
        if sample <= 1:
            sample = int(sample * df_util.height)
        df_util = df_util.sample(n=min(sample,df_util.height),seed=313)

    if upgrade!=0:
        columns = df_util.columns
        df_up = pl.read_parquet(f"outputs/RMI_2024_7_blk_grp_pop_upgrade{upgrade}.parquet").filter((pl.col("state").str.contains(state, literal=True))).with_columns(
            bldg_id = pl.col("bldg_id").cast(str),
        )
        merge = [c for c in df.columns if c not in df_up.columns or c == "bldg_id"]
        df_util = df_util.select(merge)
        df_util = df_util.join(df_up, on = "bldg_id", how="left").select(columns)
        pass

    return df_util.sort(pl.col("bldg_id").cast(int))    #adjusted with Al to to ensure bldg_id is sorted numerically not lexographically, which was causing some issues with the load profile aggregation weighting

if __name__=="__main__":
    segment = get_segment(state,utility,gas_utility) # + kwargs, u know what they are
    pass

In [ ]:
import os, io, asyncio, aioboto3, polars as pl
from botocore.config import Config
from botocore import UNSIGNED
from datetime import datetime
from tqdm.auto import tqdm

BUCKET = "oedi-data-lake"
BASE = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/timeseries_individual_buildings/by_state"
REGION = "us-west-2"  # OEDI region

COLUMNS = [
    "timestamp",
    "out.electricity.net.energy_consumption",
    "out.electricity.heating.energy_consumption",
    "out.electricity.heating_hp_bkup.energy_consumption",
    "out.electricity.cooling.energy_consumption",
    "out.natural_gas.total.energy_consumption",
    "out.natural_gas.heating.energy_consumption",
]
RENAME = {
    "electricity.total": "out.electricity.net.energy_consumption",
    "electricity.heating": "out.electricity.heating.energy_consumption",
    "electricity.secondary_heating": "out.electricity.heating_hp_bkup.energy_consumption",
    "electricity.cooling": "out.electricity.cooling.energy_consumption",
    "natural_gas.total": "out.natural_gas.total.energy_consumption",
    "natural_gas.heating": "out.natural_gas.heating.energy_consumption",
}

select_by_id = pl.selectors.matches(r"^\-?[0-9]+$|^.*_whole$")
select_quants = pl.selectors.matches(r"total_load|total_cost")
select_labels = pl.selectors.temporal()|pl.selectors.matches("id")|pl.selectors.string()
select_costs = pl.selectors.ends_with("_cost").exclude("total_cost")

LOGICAL_KEYS = list(RENAME.keys())
FUEL_ENDUSE = {k: tuple(k.split(".", 1)) for k in LOGICAL_KEYS}

FUEL_WEIGHT_PREFIX = {
    "electricity": "elec",
    "natural_gas": "gas",
}

cfg = Config(
    region_name=REGION,
    signature_version=UNSIGNED,
    retries={"max_attempts": 5, "mode": "adaptive"},
    max_pool_connections=256,
    read_timeout=120, connect_timeout=10,
)

def key_for(state, bid, upgrade):
    return f"{BASE}/upgrade={upgrade}/state={state}/{bid}-{upgrade}.parquet"

async def fetch_one(s3, key, logicals):
    obj = await s3.get_object(Bucket=BUCKET, Key=key)
    body = await obj["Body"].read()
    buf = io.BytesIO(body)
    cols_to_read = ['timestamp'] + [RENAME[logical] for logical in logicals]
    df = pl.read_parquet(buf, columns=cols_to_read)
    ts = (
        df.select("timestamp")
          .with_columns(pl.col("timestamp").dt.offset_by("-15m"))
          .with_columns(pl.col("timestamp").dt.replace(year=2025))
          .with_columns(
              pl.col("timestamp")
                .dt.replace_time_zone("UTC")
                .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
          )
          .sort("timestamp")
    )
    vals = df.drop("timestamp", strict=False).rename({RENAME[logical]:logical for logical in logicals})
    return ts, vals

CHUNK = 1000

async def fetch_building_metric(s3, state, bid, upgrade, logicals, sem):
    async with sem:
        key = key_for(state, bid, upgrade)
        ts, vals = await fetch_one(s3, key, logicals)
    renamed_vals = vals.rename({logical: f"{logical}|{bid}" for logical in logicals})
    return ts, renamed_vals


async def get_load_profiles(
    metadata: pl.DataFrame,
    fuel: list[str] | str = ["electricity"],
    end_use: str = "total",
    concurrency: int = 64,
    outdir: str = "outputs/load_profiles",
) -> dict[str, pl.DataFrame]:
    """
    Fetch ALL fuels together in one S3 call per building.
    Split and save to separate canonical files per fuel.
    Per-combo output written under utility folders with fuel subfolders.
    Return value is: { f"{fuel}_{end_use}": raw_df_for_that_fuel }
    """

    if isinstance(fuel, str):
        fuel = [fuel]
    
    for f in fuel:
        logical = f"{f}.{end_use}"
        if logical not in RENAME.keys():
            raise ValueError(f"Invalid metric {logical}")

    sem = asyncio.Semaphore(concurrency)
    logicals = [f"{f}.{end_use}" for f in fuel]
    group_cols = ["state", "elec_utility", "gas_utility", "upgrade"]

    output_by_fuel: dict[str, dict[str, pl.DataFrame]] = {
        f"{f}_{end_use}": {} for f in fuel
    }

    combo_iter = metadata.group_by(group_cols, maintain_order=True)
    for (state, eu, gu, upgrade), meta_g in tqdm(
        combo_iter,
        desc=f"Combos ({','.join(fuel)}.{end_use})",
        total=combo_iter.agg().height,
        leave=True,
    ):
        ids = meta_g["bldg_id"].unique().sort().to_list()

        canonical_dfs_by_fuel = {}
        missing_ids_by_fuel = {}
        
        for f in fuel:
            canonical_dir = os.path.join(outdir, state, "_canonical", str(upgrade), f)
            os.makedirs(canonical_dir, exist_ok=True)
            canonical_path = os.path.join(canonical_dir, f"{end_use}.parquet")
            
            if os.path.exists(canonical_path):
                canonical_dfs_by_fuel[f] = pl.read_parquet(canonical_path)
                existing_ids = {c for c in canonical_dfs_by_fuel[f].columns if c.isdigit()}
            else:
                canonical_dfs_by_fuel[f] = None
                existing_ids = set()
            
            missing_ids_by_fuel[f] = [bid for bid in ids if str(bid) not in existing_ids]

        all_missing = set()
        for missing_list in missing_ids_by_fuel.values():
            all_missing.update(missing_list)
        all_missing = sorted(list(all_missing))

        if all_missing:
            print(f"{len(all_missing)} missing IDs for combo, fetching...")
            async with aioboto3.Session().client("s3", config=cfg) as s3:
                ts_ref = None
                for f in fuel:
                    if canonical_dfs_by_fuel[f] is not None:
                        ts_ref = canonical_dfs_by_fuel[f].select("timestamp")
                        break

                fetch_pbar = tqdm(
                    total=len(all_missing),
                    desc=f"{state}-{eu}_{gu}-{upgrade} downloading",
                    leave=False,
                )

                for i in range(0, len(all_missing), CHUNK):
                    batch = all_missing[i : i + CHUNK]
                    tasks = [
                        asyncio.create_task(
                            fetch_building_metric(
                                s3,
                                state,
                                bid,
                                upgrade,
                                logicals,
                                sem,
                            )
                        )
                        for bid in batch
                    ]

                    cols_list = []

                    for fut in asyncio.as_completed(tasks):
                        try:
                            ts, col = await fut
                        except Exception as e:
                            print("skip:", e)
                            continue

                        if ts_ref is None:
                            ts_ref = ts
                        else:
                            if not ts_ref["timestamp"].equals(ts["timestamp"]):
                                raise ValueError(
                                    f"Timestamp misalignment for state={state}, "
                                    f"upgrade={upgrade}, fuels={fuel}"
                                )

                        cols_list.append(col)
                        fetch_pbar.update(1)

                    if not cols_list:
                        continue

                    new_wide = pl.concat(cols_list, how="horizontal")
                    new_wide = pl.concat([ts_ref, new_wide], how="horizontal")

                    for f in fuel:
                        fuel_col_pattern = f"{f}.{end_use}|"
                        fuel_cols_in_batch = [c for c in new_wide.columns if c.startswith(fuel_col_pattern)]
                        
                        if not fuel_cols_in_batch:
                            continue
                        
                        fuel_batch = new_wide.select(["timestamp"] + fuel_cols_in_batch)
                        rename_map = {col: col.rsplit("|", 1)[-1] for col in fuel_cols_in_batch}
                        fuel_batch = fuel_batch.rename(rename_map)
                        
                        if canonical_dfs_by_fuel[f] is None:
                            canonical_dfs_by_fuel[f] = fuel_batch
                        else:
                            canonical_dfs_by_fuel[f] = canonical_dfs_by_fuel[f].join(
                                fuel_batch, on="timestamp", how="outer", coalesce=True
                            )

                fetch_pbar.close()

                for f in fuel:
                    if canonical_dfs_by_fuel[f] is None:
                        continue
                    
                    numeric_cols = [c for c in canonical_dfs_by_fuel[f].columns if c != "timestamp" and c.isdigit()]
                    cols_sorted = ["timestamp"] + sorted(numeric_cols, key=int)
                    canonical_dfs_by_fuel[f] = canonical_dfs_by_fuel[f].select(cols_sorted)

                    canonical_dir = os.path.join(outdir, state, "_canonical", str(upgrade), f)
                    canonical_path = os.path.join(canonical_dir, f"{end_use}.parquet")
                    print(f"  Writing {len(numeric_cols)} columns for {f} to {canonical_path}...")
                    canonical_dfs_by_fuel[f].write_parquet(canonical_path)
                    import gc; gc.collect()


        for f in fuel:
            logical_key = f"{f}_{end_use}"
            canonical_df = canonical_dfs_by_fuel[f]
            
            if canonical_df is None:
                continue

            combo_bids = [str(bid) for bid in ids if str(bid) in canonical_df.columns]
            
            if not combo_bids:
                continue

            combo_raw = canonical_df.select(["timestamp"] + combo_bids)
            combo_raw = combo_raw.with_columns(pl.col("timestamp").str.to_datetime(time_zone="UTC"))

            combo_dir = os.path.join(
                outdir,
                state,
                eu if eu else "Other",
                gu if gu else "Other",
                str(upgrade),
                f,
            )
            if os.path.exists(os.path.join(combo_dir, f"{end_use}.parquet")):
                pass
            else:
                os.makedirs(combo_dir, exist_ok=True)
                combo_path = os.path.join(combo_dir, f"{end_use}.parquet")
                combo_raw.write_parquet(combo_path)

            up_key = upgrade
            output_by_fuel[logical_key][up_key] = combo_raw

    result = {}
    for logical_key, upgrade_dict in output_by_fuel.items():
        if not upgrade_dict:
            raise ValueError(f"No results produced for {logical_key}")

        try:
            dfs = [df for k, df in upgrade_dict.items()]
            if dfs:
                from functools import reduce
                merged_all = reduce(
                    lambda a,b: a.join(b, on="timestamp", how="inner").drop(pl.selectors.ends_with("_right")), dfs)
                all_cols = [c for c in merged_all.columns if c != "timestamp"]
                merged_all = merged_all.select("timestamp", *sorted(all_cols, key=lambda x: int(x)))
                upgrade_dict["all"] = merged_all
        except Exception as e:
            print(f"Warning: could not build 'all' merged dataframe for {logical_key}:", e)

        result[logical_key] = upgrade_dict["all"] if "all" in upgrade_dict else dfs[0] if dfs else None

    return result


def adjust_load_profile(load, metadata, weight_col='elec_weight', customer_count=None, segment_monthly_consumption=None):
    """
    Adjust load profiles by applying weights and scaling to match customer count and monthly consumption targets.
    
    Args:
        load (pl.DataFrame): Wide DataFrame with 'timestamp' column and building ID columns containing load values
        metadata (pl.DataFrame): DataFrame with 'bldg_id' and f'{system}_weight' columns
        customer_count (float, optional): Target customer count to scale weights to
        segment_monthly_consumption (dict, optional): Target monthly consumption {date_string: kwh_value}
    
    Returns:
        dict: {
            'adjusted_load': Wide DataFrame with adjusted load profiles,
            'monthly_summary': DataFrame with monthly aggregated consumption,
            'customer_count_scale_factor': Float,
            'monthly_scale_factors': Dict mapping months to their scale factors
        }
    """
    import pandas as pd
    
    if load.is_empty() or metadata.is_empty():
        raise ValueError("Load and metadata DataFrames cannot be empty")
    
    if "bldg_id" not in metadata.columns or weight_col not in metadata.columns:
        raise ValueError(f"Metadata must have 'bldg_id' and {weight_col} columns")
    
    meta = metadata.group_by("bldg_id", maintain_order=True).agg(pl.col(weight_col).sum())
    bldg_ids = load.select(select_by_id).columns
    if len(bldg_ids) > len(meta) != 0:
        raise ValueError("Load Dataframe numeric columns must be less than the number of building IDs in metadata DataFrame")
    else:
        meta = meta.filter(pl.col("bldg_id").cast(str).is_in(bldg_ids))
        weights = meta[weight_col]
    
    load = load.select(select_labels, select_quants, select_by_id, select_costs)
    
    original_customer_count = weights.sum()
    
    if customer_count is not None:
        customer_count_scale = customer_count / original_customer_count
        weights = weights * customer_count_scale
    else:
        customer_count_scale = 1.0

    meta = meta.with_columns(weights.alias(weight_col))
    metadata = metadata.join(meta, on="bldg_id", how="left").with_columns(
        pl.when(pl.col("bldg_id").cast(str).is_in(bldg_ids))
        .then(pl.col(weight_col)*(weights/(pl.col(weight_col).sum().over("bldg_id"))))
    ).drop(weight_col+"_right")

    weighted_load = load.with_columns(
        [pl.col(str(bid)) * weight for bid, weight in zip(bldg_ids, weights)]
    )
    
    if 'timestamp' in weighted_load.columns and 'rateName' not in weighted_load.columns:
        monthly_agg = weighted_load.group_by_dynamic("timestamp", every="1mo").agg(
            pl.sum_horizontal(select_by_id).sum().alias("kwh")
        ).sort(select_labels)
    
    if segment_monthly_consumption is not None:
        target_df = pl.DataFrame({
            "timestamp": [pd.to_datetime(k, utc=True) for k in segment_monthly_consumption.keys()],
            "target_kwh": list(segment_monthly_consumption.values())
        }).sort("timestamp")
        
        comparison = monthly_agg.join(target_df, on="timestamp", how="left")
        
        comparison = comparison.with_columns(
            (pl.col("target_kwh") / pl.col("kwh")).alias("monthly_scale_factor")
        )
        
        monthly_scale_factors = {
            row[0]: row[2] for row in comparison.select(["timestamp", "kwh", "monthly_scale_factor"]).iter_rows()
        }
        
        adjusted_load = weighted_load.with_columns(
            pl.col("timestamp").dt.truncate("1mo").alias("month_key")
        ).join(
            comparison.select([pl.col("timestamp").alias("month_key"), "monthly_scale_factor"]),
            on="month_key",
            how="left"
        ).with_columns(
            [pl.col(str(bid)) * pl.col("monthly_scale_factor") for bid in bldg_ids]
        ).drop(["month_key", "monthly_scale_factor"])
        
        monthly_summary = adjusted_load.group_by_dynamic("timestamp", every="1mo").agg(
            pl.sum_horizontal(select_by_id).sum().alias("kwh")
        ).sort("timestamp")
        detailed_summary = adjusted_load.select([
            pl.col("timestamp"),
            pl.sum_horizontal(select_by_id).alias("kwh")
        ])
    else:
        adjusted_load = weighted_load
        if 'timestamp' in weighted_load.columns and 'rateName' not in weighted_load.columns:
            monthly_summary = monthly_agg
            monthly_scale_factors = {row[0]: 1.0 for row in monthly_agg.select("timestamp").iter_rows()}
        else:
            monthly_agg = monthly_summary = monthly_scale_factors = "Unavailable"
        detailed_summary = adjusted_load.select(
            select_labels,select_quants,
            pl.sum_horizontal(select_by_id).alias("kwh")
        )
    
    return {
        'metadata': metadata,
        'adjusted_load': adjusted_load,
        'original_monthly': monthly_agg,
        'monthly_summary': monthly_summary,
        'detailed_summary': detailed_summary,
        'customer_count_scale_factor': customer_count_scale,
        'monthly_scale_factors': monthly_scale_factors
    }


In [ ]:
NON_METADATA_COLS = [
    "elec_utility",
    "elec_weight",
    "gas_utility",
    "gas_weight",
    "elec_county_coverage",
    "gas_county_coverage",
    "SP_elec",
    "SP_gas",
    "electricity.total",
    "natural_gas.total",
    "heating_pct",
    "cooling_pct",
    "hot_water_pct",
    "cooking_pct",
    "others_pct",
    "total_kwh"
]

def get_segment(state, utility="", gas_utility="", upgrade = 0, sample=None, **kwargs):
    """
    State is required, utility and gas_utility are recommended. The rest of the arguments are optional.

    Provide each segment with a *list* of options applicable to that segment.
    
    It works by finding the all entries that contain any string provided as input for a given segment. For
    example, providing "Electric" for heating_type will include all electric resistance heating systems
    (baseboards, boilers, etc). Below is a list of all available options for each segment, but there are important
    options that can be matched that don't appear in this list, such as the county or city, as well as a few others.

    "heating_type":         "Electric | Natural Gas | Propane | Oil | Shared | None", (k)\n
    "building_type":        "SF | Small MF | Large MF | Mobile", (e) \n
    "wh_type":              "Heat Pump | Electric | Natural Gas | Propane | Oil", (k)\n
    "area":                 "0-1499 | 1500-2499 | 2500-3999 | 4000+", (e)\n
    "income":               "Not Available | Low Income (<40,000) | Moderate Income (40,000-99,999) | High Income (>100,000)", (e)\n
    "climate_zone":         "Cold | Hot-Dry | Hot-Humid | Marine | Mixed-Dry | Mixed-Humid | Very Cold", (e)\n
    "vintage":              "<1980 | 1980-2000 | 2000-2010 | >2010", (e)\n
    "insulation_level":     "Good Insulation | Average Insulation | Poor Insulation", (e)\n
    "has_solar":            "Yes | No" (e)

    (e) = exact match only, (k) = keyword match

    """
    df = pl.read_parquet("outputs/RMI_2024_7.5_blk_grp_pop_cust_cnt.parquet").filter((pl.col("state").str.contains(state, literal=True)))

    # Handle utility as string or list
    if isinstance(utility, str):
        df_util = df.filter(pl.col("elec_utility").str.contains(utility, literal=True))
    else:
        df_util = df.filter(pl.col("elec_utility").str.contains_any(utility, literal=True))

    df_util = df_util.with_columns(
        bldg_id = pl.col("bldg_id").cast(str),
        zip_code = pl.col("zip_code").cast(str).str.zfill(5)
    )

    # Electric Utility name invalid for state
    if df_util.is_empty():
        raise Exception(f"Utility name invalid. Has to be one of the following: {', '.join(df['elec_utility'].drop_nulls().unique().sort())}")
    
    # Handle gas utility as string or list
    if isinstance(gas_utility, str):
        df_util = df_util.filter(pl.col("gas_utility").str.contains(gas_utility, literal=True))
    else:
        df_util = df_util.filter(pl.col("gas_utility").str.contains_any(gas_utility, literal=True))
    
    # Gas Utility name invalid for state or no buildings with the selected electric utility are served gas by the selected gas utility
    if df_util.is_empty():
        raise Exception(f"Gas utility name invalid or no buildings with the selected electric utility are served gas by the selected gas utility. Choose one of the following: {', '.join(df['gas_utility'].unique().sort().to_list())}?")

    # Filtering
    for col_name, values in kwargs.items():
        if not values:
            continue  # Skip empty values
        if col_name not in df.columns:
            raise Exception(f"{col_name} is not a valid name. Allowed names are {', '.join(df.columns)}")

        if all(~df[col_name].str.contains_any(values) if type(values[0]) is str else ~df[col_name].is_in(values)):
            raise Exception(f"One of {values} is not valid for {col_name}. Allowed options are {df[col_name].unique().sort().to_list()}")

        df_util = df_util.filter(pl.col(col_name).str.contains_any(values)) if type(values[0]) is str else df_util.filter(pl.col(col_name).is_in(values))
    
    if df_util.is_empty():
        raise Exception("Segment is too narrow for this utility and has no samples in ResStock.\nCheck sgements_by_utility folder for avilable segments for this utility.")

    if sample is not None:
        if sample <= 1:
            sample = int(sample * df_util.height)
        df_util = df_util.sample(n=min(sample,df_util.height),seed=313)

    if upgrade!=0:
        columns = df_util.columns
        df_up = pl.read_parquet(f"outputs/RMI_2024_7_blk_grp_pop_upgrade{upgrade}.parquet").filter((pl.col("state").str.contains(state, literal=True))).with_columns(
            bldg_id = pl.col("bldg_id").cast(str),
        )
        merge = [c for c in df.columns if c not in df_up.columns or c == "bldg_id"]
        df_util = df_util.select(merge)
        df_util = df_util.join(df_up, on = "bldg_id", how="left").select(columns)

    return df_util.sort(pl.col("bldg_id").cast(int))


def join_upgrade(base_segment, upgrade_mapping):
    """
    Join base segment with upgrade building IDs across multiple upgrade levels.
    
    Args:
        base_segment: Base segment DataFrame from get_segment()
        upgrade_mapping: Dict mapping upgrade_id (int) -> list of building IDs
    
    Returns:
        DataFrame with upgrade column appended and building IDs duplicated for each upgrade level
    """
    dfs = [base_segment.with_columns(pl.lit(0).alias("upgrade"))]
    
    for upgrade_id, bldg_ids in upgrade_mapping.items():
        upgrade_df = base_segment.filter(
            pl.col("bldg_id").cast(str).is_in([str(bid) for bid in bldg_ids])
        ).with_columns(pl.lit(upgrade_id).alias("upgrade"))
        dfs.append(upgrade_df)
    
    return pl.concat(dfs)


In [ ]:
select_by_id = pl.selectors.matches(r"^\-?[0-9]+$|^.*_whole$")
select_quants = pl.selectors.matches(r"total_cost")
select_costs = pl.selectors.ends_with("_cost").exclude("total_cost")

def batch_agg_by_period(df, every, period=None, batch_size=500, op ="sum"):
    """Fast aggregation for wide DataFrames: groups by time and sums numeric columns in batches.
    
    Processes numeric columns in batches to avoid Polars creating thousands of separate sum operations.
    Useful for DataFrames with many building ID columns (15k+).
    
    Args:
        df: DataFrame with timestamp + numeric (building ID) columns
        every: Period string (e.g., "1mo", "1d", "1h")
        period: Optional period parameter for group_by_dynamic
        batch_size: Number of numeric columns per batch (default 500)
        op: Polars aggregation function as string (default "sum", can be "max", "mean", etc.)
    
    Returns:
        Aggregated DataFrame with same structure (timestamp + summed numeric columns)
    """
    cols = df.select(select_by_id|select_quants|select_costs).columns
    
    if not cols:
        return df
    
    results = []
    for i in range(0, len(cols), batch_size):
        batch_cols = cols[i:i+batch_size]
        batch_df = df.select(["timestamp"] + batch_cols)
        func = {
            "sum": pl.all().exclude("timestamp").sum(),
            "max": pl.all().exclude("timestamp").max(),
            "mean": pl.all().exclude("timestamp").mean(),
            "min": pl.all().exclude("timestamp").min()
        }
        if period:
            agg = batch_df.group_by_dynamic("timestamp", every=every, period=period).agg(
                func[op]
            )
        else:
            agg = batch_df.group_by_dynamic("timestamp", every=every).agg(
                func[op]
            )
        
        results.append(agg)
    
    # Join all batches back together on timestamp
    final = results[0]
    for r in results[1:]:
        final = final.join(r, on="timestamp", how="inner")
    
    return final.select(["timestamp",*cols]).sort("timestamp")

@dataclass
class charge_type():
    unit: str = ""

    def __call__(self, load):
        id_cols = load.select(select_by_id).columns

        if self.unit in ["month", "bill"]:
            load = batch_agg_by_period(load, "1mo")
            load = load.with_columns([pl.lit(1.0, dtype=pl.Float32).alias(c) for c in id_cols])
            
        elif self.unit in ["day"]:
            load = batch_agg_by_period(load, "1d")
            load = load.with_columns([pl.lit(1.0, dtype=pl.Float32).alias(c) for c in id_cols])
            
        elif self.unit in ["year"]:
            load = batch_agg_by_period(load, "1y")
            load = load.with_columns([pl.lit(1.0, dtype=pl.Float32).alias(c) for c in id_cols])

        elif self.unit in ["therm", "ccf"]:
            load = load.with_columns(
                (select_by_id*0.03412).round(9) if self.unit=="therm" else (select_by_id/30.2).round(9) if self.unit=="ccf" else select_by_id.round(9)
            )
        else:
            pass
        return load

@dataclass
class seasonal():
    season: str = ""

    def __call__(self, load):
        mo1, day1 = map(int, self.season.split("-")[0].split("/"))
        mo2, day2 = map(int, self.season.split("-")[1].split("/"))
        t1, t2 = (datetime(2025,mo1,day1,tzinfo=timezone.utc),datetime(2025,mo2,day2,tzinfo=timezone.utc))
        if mo1 <= mo2:
            t2 += timedelta(hours=23,minutes=59)
            filter = pl.col("timestamp").is_between(t1,t2)
        else:
            t1 -= timedelta(minutes=1)
            t2 += timedelta(days=1)
            filter = ~pl.col("timestamp").is_between(t2,t1)
        
        load = load.with_columns(
            pl.when(filter).then(select_by_id).otherwise(pl.lit(0.0))
        )

        return load

@dataclass
class tou():
    tou: str = ""

    def __call__(self, load):
        if eval(self.tou):
            filter = False
            tou = eval(self.tou)
            for t_d, t_h in tou:
                start_day, end_day = t_d
                start_time, end_time = t_h
                if start_time < end_time:
                    filter = filter | ((pl.col("timestamp").dt.hour() >= start_time) & (pl.col("timestamp").dt.hour() < end_time) &
                                    (pl.col("timestamp").dt.weekday()>= start_day) & (pl.col("timestamp").dt.weekday() <= end_day))
                else:
                    filter = filter | (((pl.col("timestamp").dt.hour() >= start_time) | (pl.col("timestamp").dt.hour() < end_time)) &
                                    (pl.col("timestamp").dt.weekday()>= start_day) & (pl.col("timestamp").dt.weekday() <= end_day))
            load = load.with_columns(
                pl.when(filter).then(select_by_id).otherwise(pl.lit(0.0))
            )
        return load

@dataclass
class tiered():
    start: float = 0.0
    end: float = float("inf")
    freq: str = "1mo"
    
    def __call__(self,load):
        self.start = float(self.start)
        self.end = float(self.end)
        load = batch_agg_by_period(load, self.freq)
        load = load.with_columns((select_by_id - self.start).clip(0,self.end-self.start))
        return load

@dataclass
class demand():
    unit: str = "30min"

    def __call__(self, load):
        if "60min" in self.unit:
            load = batch_agg_by_period(load, "15m", period="1h")
            load = batch_agg_by_period(load, "1mo", func="max")
        elif "30min" in self.unit:
            load = batch_agg_by_period(load, "15m", period="30m")
            load = batch_agg_by_period(load, "1mo", func="max")
            load = load.with_columns(select_by_id*2)
        else:
            load = batch_agg_by_period(load, "1mo", func="max")
            load = load.with_columns(select_by_id*4)
        return load

@dataclass
class Rate:
    id: int = 0
    name: str = ""
    rate: dict = field(default_factory=dict)
    category: str = ""
    unit: str = ""
    season: str = "01/01-12/31"
    start: float = 0.0
    end: float = float("inf")
    tou: str = "[]"
    tier_freq: str = "1mo"
    applied_to: str = ""
    metadata: pl.DataFrame = field(default_factory=pl.DataFrame)
    custom: callable = None

    load_components: tuple[callable, ...] = field(default_factory=tuple)
    cost_components: tuple[callable, ...] = field(default_factory=tuple)

    def __post_init__(self):
        self.unit = self.unit.split("per ")[-1].lower()
        self.tier_freq = "1d" if "day" in self.tier_freq else "1mo"
        self.category = "FIXED_M" if self.unit in ["bill", "month"] else "FIXED_D" if self.unit=="day" else "FIXED_Y" if self.unit=="year" else self.category

        if not self.custom:
            self.load_components = [charge_type(self.unit)]

            self.load_components += [
                seasonal(self.season),
                tou(self.tou)
            ]
            
            if self.unit in ["60min kw", "30min kw", "kw"]:
                self.load_components += [demand(self.unit)]
            
            self.load_components += [tiered(self.start,self.end,self.tier_freq)] if self.unit not in ["bill", "month", "day", "year", "percent"] else []

        else:
            self.load_components = [self.custom]

    def process_load(self, profile):

        for component in self.load_components:
            profile = component(profile)
        
        return profile
    
    def monthly_cost(self, orig_load):
        """
        Pass timestamp and load as a 2-column df
        """
        load = self.process_load(orig_load)
        load = batch_agg_by_period(load, "1mo")

        cost_profile = load.with_columns([
            (pl.col(col) * self.rate["rate"]).round(9).alias(f"{col}_cost") 
            for col in load.select(select_by_id).columns
        ])
        cost_profile = cost_profile.select([
            pl.col("timestamp").alias("timestamp"),
            pl.lit(self.id).alias("id"),
            pl.lit(self.name).alias("rateName"),
            pl.lit(self.category).alias("category"),
            pl.sum_horizontal(select_costs).round(9).alias("total_cost"),
            select_by_id.round(9),
            select_costs.round(9)
        ])

        return cost_profile

    def process_cost_riders(self, profile):

        self.category = "COST"
        profile = self.process_load(profile)        
        result = profile.with_columns(select_costs*(self.rate["rate"]/100).round(9))

        return result.select([
            pl.col("timestamp").alias("timestamp"),
            pl.lit(self.id).alias("id"),
            pl.lit(self.name).alias("rateName"),
            pl.lit(self.category).alias("category"),
            pl.sum_horizontal(select_costs).round(9).alias("total_cost"),
            select_by_id.round(9),
            select_costs.round(9)
        ])

class Tariff:
    def __init__(self, tariff, metadata = pl.DataFrame(), use_weights=False, custom_rates = {}):
        self.use_weights = use_weights          # customer counts flag
        self.metadata = metadata                # all info about buildings being processed
        self.tariff = tariff
        if "id" not in tariff.columns:
            tariff = tariff.select([pl.int_range(0,pl.len()).alias("id"),pl.all()])
        else:
            tariff = tariff.with_columns(pl.col("id"))
        rates = self.parse_tariff(tariff, custom_rates)       # parses all the rates from input tariff

        # separates load based rates from cost based rates 
        self.consumption_rates = [r for r in rates if "percent" not in r.unit]
        self.cost_rates = [r for r in rates if "percent" in r.unit]
        
        if any("percent" in r.unit for r in rates) and all(r.applied_to=="" for r in self.cost_rates):
            print(f"WARNING! Detected cost-based riders but applied_to column is empty, excluding... Populate applied_to column to include cost-based riders.")
        
        # Determine which weight column to use based on consumption rate units
        # If kWh or kW exists: use elec_weight; if therm or ccf exists: use gas_weight
        self.weight_col = None
        if not metadata.is_empty():
            units = [r.unit for r in self.consumption_rates]
            if any(u in ["kwh", "kw", "60min kw", "30min kw"] for u in units):
                self.weight_col = "elec_weight"
            elif any(u in ["therm", "ccf"] for u in units):
                self.weight_col = "gas_weight"
            
    def parse_tariff(self, tariff, custom_rates):
        if len([k for k in tariff.columns if "/" in k])==12:
            pass
        else:
            raise ValueError("Invalid tariff structure")
        rates=[]
        for rate in tariff.iter_rows(named=True):
            # Skip location based rates
            if rate.get("Location") or not rate.get("Rate Determinant"):
                continue

            # Collect rate constraints
            mapping = {
                "id": "id",
                "name": "Component Description",
                "rate": "Rate",
                "category": "Category",
                "unit": "Rate Determinant",
                "season": "Season",
                "start": "Start",
                "end": "End",
                "tou": "tou",
                "tier_freq": "Determinant",
                "applied_to": "applied_to"
            }
            kwags = {k: rate.get(v, "") for k,v in mapping.items() if rate.get(v) not in [None,""]}

            # Collect rates
            kwags["rate"] = pl.DataFrame(
                {
                    "month": [int(k.split("/")[0]) for k,_ in rate.items() if "/" in k],
                    "rate": [v for k,v in rate.items() if "/" in k]
                }
            ).sort("month")

            # Propogate metadata for customer counts
            if self.use_weights:
                kwags["metadata"] = self.metadata

            # Check for custom rate handling
            for ids, func in custom_rates.items():
                if kwags["id"] in ids:
                    kwags["custom"] = func
                    break

            rates.append(Rate(**kwags))
        
        return rates

    def annual_bill(self, load_orig):
        # Format load and optimize for non-demand and non-tou rates
        load = pl.DataFrame(load_orig)

        # Collect monthly totals
        monthly = []
        print("Processing consumption rates...")
        for r in tqdm(self.consumption_rates):
            if r.unit in ["60min kw", "30min kw", "kw"]:
                processed_bills = r.monthly_cost(load)
                monthly.append(processed_bills)
            elif eval(r.tou):
                temp = batch_agg_by_period(load, "1h")
                processed_bills = r.monthly_cost(temp)
                monthly.append(processed_bills)
            elif r.tier_freq=="day":
                temp = batch_agg_by_period(load, "1d")
                processed_bills = r.monthly_cost(temp)
                monthly.append(processed_bills)
            else:
                temp = batch_agg_by_period(load, "1mo")
                processed_bills = r.monthly_cost(temp)
                monthly.append(processed_bills)

        # Apply cost rates
        agg_costs = pl.concat(monthly)
        print("Processing cost-based riders...")
        percent_costs = [
            r.process_cost_riders(
                agg_costs
                .filter(pl.col("id").cast(str).is_in(r.applied_to.split(",")))
                .group_by("timestamp")
                .agg(select_quants.sum(),select_by_id.sum(),select_costs.sum())
            )
            for r in tqdm(self.cost_rates) if r.applied_to
        ]
        monthly += percent_costs

        # Apply weights if you want - changes total costs based on customer counts
        monthly = pl.concat(monthly)
        if self.use_weights and self.weight_col is not None:
            # Use adjust_load_profile to apply weights to calculate aggregations
            monthly = adjust_load_profile(monthly, self.metadata)['adjusted_load']
            # Adjust totals
            monthly = monthly.with_columns(
                pl.sum_horizontal([pl.col(c) for c in monthly.select(select_costs).columns]).round(9).alias("total_cost")
            )

        # Collect results
        results = {}

        results['load'] = monthly.group_by("timestamp").agg(select_by_id.first()).sort("timestamp")
        
        monthly = monthly.select(
            pl.col("id","rateName","category","timestamp"), select_quants,
            select_costs.name.map(lambda c: c.replace("_cost", ""))
        )

        # Annual Bill
        results["annual"] = (
            monthly.select(
                select_quants.sum().round(9),
                select_by_id.sum().round(9)).sum()
        )

        # Breakdown by Rate
        results["by_rate"] = (
            monthly
                .group_by(["id","rateName","category"],maintain_order=True)
                .agg(
                    select_quants.sum().round(9),
                    select_by_id.sum().round(9)
                )
                .sort([pl.col("id").cast(int),"rateName","category"])
        )
        
        # Final Monthly
        results["monthly"] = (
            monthly
                .group_by("timestamp",maintain_order=True)
                .agg(
                    select_quants.sum().round(9),
                    select_by_id.sum().round(9)
                )
                .sort(["timestamp"])
        )

        # Detailed monthly
        results["detailed"] = (
            monthly
                .group_by(["id","rateName","category","timestamp"],maintain_order=True)
                .agg(
                    select_quants.sum().round(9),
                    select_by_id.sum().round(9)
                )
                .sort([pl.col("id").cast(int),"rateName","category","timestamp"])
        )

        return Bill("electric" if "elec" in self.weight_col else "gas", "", results, self.metadata)


In [ ]:
import requests
import glom

def get_elec_tariff(elecTariff, utility, building, territory="", custom_path=""):
    if custom_path:
        return custom_path.rsplit('_',1)[0], pl.read_csv(custom_path)

    territoryId, territoryName = territory if territory else ("","")

    # --- CHANGED: build a safe dir path and ensure it exists
    dir_path = os.path.join("outputs", "electric_tariffs", utility)
    os.makedirs(dir_path, exist_ok=True)
    
    search = f"{territoryName}_{elecTariff}" if territoryName else f"{elecTariff}"
    files = [f for f in os.listdir(dir_path) if f.endswith(search+".csv") and not f.startswith("._")]
    if files:
        file_path = os.path.join(dir_path, files[0])
        df = pl.read_csv(file_path)
        df = df.fill_null("")
        return files[0].rsplit("_",1)[0], df

    app_id = "3df8e135-968d-4399-9879-2a1c6a3de30c"
    app_key = "e51974c7-996b-4698-9628-71950d223364"

    url = "https://api.genability.com/rest/v1/ondemand/calculate"
    params = {
        "masterTariffId": elecTariff,
        "fromDateTime": building["timestamp"].first().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "toDateTime": building["timestamp"].last().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "groupBy": "MONTH",
        "propertyInputs" : [{
            "keyName": "consumption",
            "unit": "kWh",
            "fromDateTime": building["timestamp"].first().strftime("%Y-%m-%dT%H:%M:%SZ"),
            "duration": 900000, # 15 mins
            "dataSeries": building[:,1].to_list()
        }]
    }
    if territoryId:
        params["propertyInputs"].append({"keyName": "territoryId", "dataValue": territoryId})

    response = requests.post(url, auth=(app_id, app_key), json=params)
    data = response.json()["results"][0]
    tariff_name = data["tariffName"] + "-" + territoryName if territoryName else data["tariffName"]
    api_calc = pl.from_dicts(response.json()["results"][0]["items"])
    api_calc = api_calc.sort("rateName")

    # Call Tariff API
    url = "https://api.genability.com/rest/public/tariffs"

    params = {
        "masterTariffId": elecTariff,
        "fromDateTime": building["timestamp"].first().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "toDateTime": building["timestamp"].last().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "populateRates": True
    }

    response = requests.get(url, auth=(app_id, app_key), params=params)
    lseId = response.json()["results"][0]["lseId"]
    data = [d for d in response.json()["results"][0]["rates"] if d.get("territory",{}).get("territoryId",territoryId)==territoryId]
    from_tariff = pl.DataFrame({"rateName":[d["rateName"] for d in data],
                                "chargeClass": [d.get("chargeClass","") for d in data],
                                "blocks":[d["rateBands"] if len(d.get("rateBands"))>1 and any(x.get("hasConsumptionLimit",False) for x in d.get("rateBands",{})) else None for d in data ],
                                "touId": [d.get("timeOfUse",{}).get("touId") for d in data],
                                "seasonId": [d.get("season",{}).get("seasonId") for d in data]
                                })

    # Combine Tariff and Calculate API returns
    cols = [col for col in ["touId","seasonId"] if col in api_calc.columns]
    costs_breakdown = api_calc.join(from_tariff, on=["rateName","chargeClass"], how="left")
    costs_breakdown = costs_breakdown.with_columns([
        pl.coalesce(c,c+"_right").alias(c) for c in cols if c+"_right" in api_calc.columns
    ]).drop(pl.selectors.ends_with("_right"))
    costs_breakdown = (
        costs_breakdown
        .filter(pl.col("fromDateTime").str.contains(str(building["timestamp"].last().year)))
        .with_columns(fromDateTime = pl.col("fromDateTime").str.to_datetime(time_zone="UTC"))
    )
    cols = [c for c in ["rateName","chargeClass","chargeType","quantityKey","rateType","seasonId","touId","blocks"] if c in costs_breakdown.columns]
    costs_breakdown = costs_breakdown.filter(pl.col("fromDateTime").dt.year() == building["timestamp"].last().year).group_by(cols).agg(
        rates = pl.struct([pl.col("rateAmount"),pl.col("fromDateTime").dt.strftime("%m/%d/%Y")])
    ).sort("rateName")

    # Generate tariff
    rows = []
    months = [f"{str(m).zfill(2)}/01/{building['timestamp'].last().year}" for m in range(1,13)]
    for i,rate in enumerate(costs_breakdown.iter_rows(named=True)):
        rate_name = rate["rateName"]
        category = rate.get("chargeClass", "")
        determinant = ""

        # Add missing months with zero rate value (when applicable)
        rates = []
        it = iter(months)
        for month in it:
            r = rate.get("rates")
            if month in [m["fromDateTime"] for m in r]:
                rates.append([m for m in r if m["fromDateTime"] == month][0]["rateAmount"])
            else:
                rates.append(0)

        # Rate Determinant logic
        if rate.get("chargeType") == "FIXED_PRICE":
            rate_determinant = "per month"
            category = "DISTRIBUTION"
        elif rate.get("chargeType") == "QUANTITY" and rate.get("rateType")=="PERCENTAGE":
            rate_determinant = "percent"
            category = "COST"
        elif rate.get("chargeType") == "DEMAND_BASED":
            if "60min" in rate["quantityKey"]:
                rate_determinant = "per 60min kw"
            elif "ratchet" in rate["quantityKey"]:
                rate_determinant = "per kw"
            else:
                rate_determinant = "per 30min kw"
        elif rate.get("chargeType") == "CONSUMPTION_BASED":
            rate_determinant = "per kwh"
        else:
            continue

        # Season Logic
        season = ""
        if rate.get("seasonId"):
            url = f"https://api.genability.com/rest/public/seasons"
            params = {
                "lseId": lseId
            }

            response = requests.get(url, auth=(app_id, app_key), params=params).json()
            spec = ("results", ["seasons"])
            flat = glom.flatten(glom.glom(response,spec))
            for s in flat:
                if s["seasonId"]==rate.get("seasonId"):
                    season = f'{s["seasonFromMonth"]:02d}/{s["seasonFromDay"]:02d}-{s["seasonToMonth"]:02d}/{s["seasonToDay"]:02d}'
                    break
        
        # Time of Use Logic
        tou=[]
        tou_type = ""
        if rate.get("touId"):
            url = f"https://api.genability.com/rest/public/timeofuses/{rate.get('touId')}"

            response = requests.get(url, auth=(app_id, app_key)).json()
            
            for p in response["results"][0]["touPeriods"]:
                p["toHour"] = p["toHour"] if p["toHour"] else 24
                tou.append(([p["fromDayOfWeek"]+1,p["toDayOfWeek"]+1],[p["fromHour"], p["toHour"]]))
            tou_type = response["results"][0].get("touType", "OFF_PEAK")
        tou=str(tou)
        
        # Block Logic
        if rate.get("blocks"):
            bands = rate.get("blocks")
            if any(b.get("consumptionUpperLimit") for b in bands):
                prev_limit = None
                for band in bands:
                    start = prev_limit if prev_limit else ""
                    end = band.get("consumptionUpperLimit","")
                    credit = -1 if band.get("isCredit") else 1
                    rates = [credit*band.get("rateAmount") if r!=0 else 0 for r in rates]
                    determinant = "per month"
                    rows.append([i, tariff_name, rate_name, category, rate_determinant, start, end, determinant, season, tou, tou_type, ""] + rates)
                    prev_limit = end
                continue

        rows.append([i, tariff_name, rate_name, category, rate_determinant, "", "", determinant, season, tou, tou_type, ""] + rates)

    df = pl.DataFrame(rows, schema=[
        "id","tariff","Component Description", "Category", "Rate Determinant",
        "Start", "End", "Determinant", "Season", "tou", "period", "applied_to", *months
    ],orient="row")

    # Sort by charge types in order of demand, kwh, fixed, all else
    df = df.with_columns(
        sort_level = (
            pl.when(pl.col("Rate Determinant").str.contains(r"\bkw\b", literal=False))
            .then(1)
            .when(pl.col("Rate Determinant").str.contains(r"\bkwh\b", literal=False))
            .then(2)
            .when(pl.col("Rate Determinant").str.contains(r"day|month|year|bill", literal=False))
            .then(3)
            .otherwise(4)
        )
    )

    df = df.sort(["sort_level", "Component Description", "Category"])

    df = df.with_row_index("row_idx")

    group_ids = (
        df.group_by(["Component Description", "Category"])
        .agg(pl.col("row_idx").min().alias("id_new"))
        .with_columns((pl.col("id_new") + 1))
    )

    df = (
        df.drop("id")
        .join(group_ids, on=["Component Description", "Category"], how="left")
        .drop(["sort_level", "row_idx"])
        .rename({"id_new": "id"})
        .select(pl.col("id"),pl.all().exclude("id"))
    )

    df.write_csv(f"outputs/electric_tariffs/{utility}/{tariff_name}_{elecTariff}.csv")

    return tariff_name, df


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service

def get_gas_tariff(state, utility, schedule):
    # make sure schedule name can be saved as filename
    clean = lambda x: (
        x.replace("/", "-")
                .replace("\\", "-")
                .replace(" ", "_")
                .replace(">", "greater_than")
                .replace("<", "less_than")
                .replace(":", "_")
                .replace('"', "_")
                .replace("|", "_")
                .replace("?", "_")
                .replace("*", "_")
                .replace("---", "")
                .strip()
    )
    # only download if not already done
    if os.path.exists(f"outputs/gas_tariffs/{state}-{utility}/{clean(schedule)}.csv"):
        return pl.read_csv(
            f"outputs/gas_tariffs/{state}-{utility}/{clean(schedule)}.csv"
        )

    download_path = os.path.join(os.getcwd(), "outputs", "gas_tariffs")

    # Configure Chrome WebDriver
    # Al's original code was for Edge, switched to Chrome for system compatibility
    chrome_options = Options()
    chrome_options.add_argument("--headless") # disable if you want to see the browser

    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36")
    chrome_options.add_argument("--log-level=3")
    chrome_options.add_experimental_option("useAutomationExtension", False)  # Disable automation extension
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation", "enable-logging"])  # Remove automation flag
    chrome_options.add_experimental_option("prefs", {
    "download.default_directory": download_path,
    "download.prompt_for_download": False,
    "directory_upgrade": True
    })

    service = Service(log_path=os.devnull)

    # Initialize Chrome WebDriver
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    LINK = "https://secure.rateacuity.com/RateAcuityPortal/Account/Login"
    
    # Navigate to login page
    driver.get(LINK)

    # Check if the 'Log off' element exists and click it if present
    try:
        logoff_element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//a[@href=\"javascript:document.getElementById('logoutForm').submit()\"]"))
        )
        logoff_element.click()
    except:
        pass  # If the element doesn't exist, continue without error
    
    # Login configuration
    EMAIL_ADDRESS = "al.qarooni@rmi.org"
    PASSWORD = "Power200"

    # Login to the page
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'UserName'))).send_keys(EMAIL_ADDRESS)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'Password'))).send_keys(PASSWORD)
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//input[@type='submit' and @value='Log in']"))
    ).click()

    # Click on 'Rate Acuity Gas Reports' link
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//a[contains(normalize-space(text()), 'Rate Acuity Gas Reports')]"))
    ).click()

    # Select History Option
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//input[@id='report' and @value='history']"))
    ).click()

    # Select a state from the dropdown
    state_dropdown = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'StateSelect')))
    select = Select(state_dropdown)
    select.select_by_value(state)

    # Get utilities list
    utility_dropdown = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'UtilitySelect')))
    utility_options = utility_dropdown.find_elements(By.TAG_NAME, 'option')
    option_texts = [option.text.strip() for option in utility_options]
    if utility not in option_texts:
        raise ValueError(f"Gas utility name invalid. Options are: {option_texts}")
    select = Select(utility_dropdown)
    select.select_by_visible_text(utility)

    # Get schedules list
    schedule_dropdown = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'ScheduleSelect')))
    schedule_options = schedule_dropdown.find_elements(By.TAG_NAME, 'option')
    option_texts = [option.text.strip() for option in schedule_options if 'res' in option.text.strip().lower() or 'multi' in option.text.strip().lower()]
    if schedule not in option_texts:
        raise ValueError(f"Choose one of the following schedules: {option_texts}")
    select = Select(schedule_dropdown)
    select.select_by_visible_text(schedule)

    ncomp = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "NComp")))
    ncomp.clear()
    ncomp.send_keys("12")

    nfreq = driver.find_element(By.ID,"NFreq")
    nfreq.clear()
    nfreq.send_keys("1")

    driver.find_element(By.XPATH, "//input[@type='submit' and @value='Search']").click()
    time.sleep(10)

    rate_file = [f for f in os.listdir(download_path) if f.endswith('.xlsx') and not f.startswith("~$")]

    if not rate_file:
        driver.quit()
        raise Exception("RateAcuity download failed, no file found in download directory.")

    rate_file_path = os.path.join(download_path, rate_file[0])

    directory_name = f"{state}-{utility}"
    target_dir = os.path.join(download_path, directory_name)
    if not os.path.exists(target_dir):
        os.makedirs(target_dir, exist_ok=True)

    # --- Load Excel file with Polars ---
    # --- Detect "Component Description" in the first column and set as header row ---
    raw_data = pl.read_excel(rate_file_path, engine="calamine", has_header=False)
    header_row_index = None

    for i, row in enumerate(raw_data.iter_rows()):
        if "Component Description" in row[0] or [utility in r for r in row[0]]:
            header_row_index = i
            break

    if header_row_index is None:
        raise Exception(f"Could not parse gas tariff {schedule}, downloaded excel file is in an unknown format.")
    
    # Organize Tariffs
    df = pl.read_excel(
        rate_file_path,
        engine="calamine",
        read_options={"header_row": header_row_index}
    )
    df = df.with_row_index("row_idx").rename({df.columns[0]: "Component Description"})

    # Add ids   
    group_ids = (
        df.group_by("Component Description")
        .agg(pl.col("row_idx").min().alias("id_new"))
        .with_columns((pl.col("id_new") + 1))
    )

    df = (
        df
        .join(group_ids, on="Component Description", how="left")
        .drop("row_idx")
        .rename({"id_new": "id"})
        .select(pl.col("id"),pl.all().exclude("id"))
    )

    df = df.select(
        ~pl.selectors.matches(r"/"),
        pl.lit("").alias("applied_to"),
        pl.selectors.matches(r"/")
    )
    if "Location" in df.columns:
        df = df.filter(pl.col("Location").is_null() | (pl.col("Location") == ""))

    df.write_csv(os.path.join(target_dir,clean(schedule) + ".csv"))
    os.remove(rate_file_path) # Remove the original file after processing

    return df

## 8. Integration & Usage Guide

### Module Dependency Map

```
┌─ segments.py
│  └─ get_segment() → Filters building metadata by utility & characteristics
│     └─ join_upgrade() → Combines base + upgrade scenarios
│
├─ load_profiles.py
│  ├─ get_load_profiles() → Async S3 fetch from OEDI data lake
│  └─ adjust_load_profile() → Apply weights & scale consumption
│
├─ genability_hack.py
│  └─ get_elec_tariff() → Fetch electric tariffs from Genability API
│
├─ rate_acuity 1.py
│  └─ get_gas_tariff() → Scrape gas tariffs via Selenium (RateAcuity portal)
│
└─ tariff_object.py
   ├─ Rate class → Individual rate component (charge_type, seasonal, tou, tiered, demand)
   ├─ Tariff class → Parse tariff CSV + calculate bills
   └─ annual_bill() → Returns Bill object with cost breakdowns
```

### Key Data Structures

| Module | Input | Output | Notes |
|--------|-------|--------|-------|
| `get_segment()` | state, utility names | pl.DataFrame | Building metadata with bldg_id, weights |
| `get_load_profiles()` | metadata DataFrame | dict[str, pl.DataFrame] | Wide format: timestamp + building ID columns |
| `adjust_load_profile()` | load, metadata | dict with 'adjusted_load' | Applies weights, scales to customer count |
| `get_elec_tariff()` | tariff ID, building load | pl.DataFrame | Standardized tariff with 12 monthly rate columns |
| `get_gas_tariff()` | state, utility, schedule | pl.DataFrame | Gas tariff from RateAcuity portal |
| `Tariff.annual_bill()` | load DataFrame | Bill object | Contains: annual, by_rate, monthly, detailed |

### Typical Workflow

1. **Define Segment** → `get_segment(state, utility_name, **filters)`
2. **Create Upgrade Mapping** → `join_upgrade(base_segment, {1: [bid1, bid2], 2: [bid3, bid4]})`
3. **Fetch Load Profiles** → `await get_load_profiles(metadata)`
4. **Get Tariffs** → `get_elec_tariff()` + `get_gas_tariff()`
5. **Calculate Bills** → `Tariff(tariff_df).annual_bill(load_df)` → Bill object

### Missing Dependencies (To Be Implemented)

These are referenced but not fully implemented:
- **`utils.bill_object.Bill`** — Stub provided; add detailed reporting methods
- **`utils.load_profiles`** — Full module already included; just needs proper utils structure
- **Annual Report Generation** — Add methods to export Bill results to Excel/PDF

### Data Checkpoint Export

After each major step, export intermediate state:
```python
# After load profile fetching
for fuel, df in load_profiles.items():
    df.write_parquet(f"checkpoints/{fuel}_profiles.parquet")

# After segment creation
segment_df.write_csv("checkpoints/segment_metadata.csv")

# After tariff calculation
bill_results['by_rate'].write_csv("checkpoints/costs_by_rate.csv")
```


## 7. RateAcuity API Module (rate_acuity 1.py)

Automates gas tariff downloads from RateAcuity portal using Selenium. Parses Excel files and outputs standardized tariff CSVs. Handles residential and multi-family service schedules.


## 6. Genability API Module (genability_hack.py)

Fetches electric tariffs from Genability API, processes them via their calculation engine, and outputs standardized tariff CSVs. Handles seasonal, TOU, and block rate structures.


## 5. Tariff Object Module (tariff_object.py)

Core tariff calculation engine. Defines Rate classes, tariff parsing, and annual bill calculation with support for complex rate structures (tiered, seasonal, TOU, demand charges).


## 4. Segments Module (segments.py)

Filters building stock metadata by state, utility, and building characteristics. Provides segment filtering and sampling.


## 3. Load Profiles Module (load_profiles.py)

Fetches and processes building load profile data from OEDI S3 bucket. Handles async concurrent fetching, canonical file caching, and load adjustment by weights.


In [ ]:
# STUB: Bill object (from utils.bill_object)
# This is a placeholder - replace with your actual Bill class implementation
from dataclasses import dataclass, field

@dataclass
class Bill:
    """Placeholder Bill class for holding tariff calculation results."""
    fuel_type: str = ""
    name: str = ""
    results: dict = field(default_factory=dict)
    metadata: pl.DataFrame = field(default_factory=pl.DataFrame)
    
    def __repr__(self):
        return f"Bill(fuel={self.fuel_type}, results_keys={list(self.results.keys())})"


# STUB: adjust_load_profile (from load_profiles.py - needs to be imported from module)
def adjust_load_profile(load, metadata, weight_col='elec_weight', customer_count=None, segment_monthly_consumption=None):
    """
    Placeholder for adjust_load_profile - returns the adjusted load profile.
    This function is also defined in load_profiles.py below - use that version.
    """
    return {
        'adjusted_load': load,
        'monthly_summary': pl.DataFrame(),
        'customer_count_scale_factor': 1.0,
        'monthly_scale_factors': {}
    }


## 2. Missing Dependencies - Stub Implementations

These are placeholder implementations for external dependencies that your modules reference. These stubs allow the code to run without errors.
